# Feature Engineering for Network Intrusion Detection

This notebook performs feature engineering on the CSE-CIC-IDS2018 dataset.

## Objectives:
1. Load and preprocess raw network flow data
2. Handle missing values and outliers
3. Create derived features
4. Encode categorical variables
5. Scale numerical features
6. Handle class imbalance
7. Save processed features for model training

In [12]:
import os
import sys
sys.path.append('..')

from pathlib import Path
import numpy as np

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import StandardScaler, VectorAssembler, StringIndexer
from pyspark.ml import Pipeline

# Stop any existing Spark session to avoid connection issues
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("Stopped existing Spark session")
except:
    pass

# Create new Spark session
spark = SparkSession.builder \
    .appName("NetworkIntrusionFeatureEngineering") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"✓ Spark initialized successfully")
print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


Stopped existing Spark session
✓ Spark initialized successfully
Spark Version: 3.5.6
Spark UI: http://ip-192-168-1-142.eu-west-1.compute.internal:4040


In [2]:
# Start timing the entire notebook execution
import time
from datetime import timedelta

notebook_start_time = time.time()
print("⏱️  Starting feature engineering pipeline...")
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")


⏱️  Starting feature engineering pipeline...
Start time: 2026-03-01 00:36:46


## 1. Load Raw Data

In [13]:
# Load data with PySpark
import sys
project_root = Path().resolve()
sys.path.insert(0, str(project_root))

from src.schemas.cse_cic_ids2018 import get_spark_schema

data_path = project_root / 'data' / 'raw' / 'CSE-CIC-IDS2018'

# Explicit schema: bypasses inferSchema scan and enforces correct types.
# Embedded duplicate header rows (e.g. "Dst Port" in a LongType column)
# fail to cast → become null → filtered out below.
df = spark.read.csv(
    str(data_path),
    header=True,
    schema=get_spark_schema(),
)

# Drop embedded header rows: they produce null in every numeric column
df = df.filter(col("Flow Duration").isNotNull())

print(f"✓ Data loaded with explicit schema")
print(f"Number of columns: {len(df.columns)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")


✓ Data loaded with explicit schema
Number of columns: 80
Partitions: 54


In [14]:
# Check data types and schema
print("Data Schema:")
df.printSchema()

print(f"\n✓ Schema loaded, ready for processing")
print(f"Partitions: {df.rdd.getNumPartitions()}")
print("Note: Skipping sample data display to avoid triggering full file scan")


Data Schema:
root
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: string (nullable = true)
 |-- Flow Duration: long (nullable = true)
 |-- Tot Fwd Pkts: long (nullable = true)
 |-- Tot Bwd Pkts: long (nullable = true)
 |-- TotLen Fwd Pkts: long (nullable = true)
 |-- TotLen Bwd Pkts: long (nullable = true)
 |-- Fwd Pkt Len Max: double (nullable = true)
 |-- Fwd Pkt Len Min: double (nullable = true)
 |-- Fwd Pkt Len Mean: double (nullable = true)
 |-- Fwd Pkt Len Std: double (nullable = true)
 |-- Bwd Pkt Len Max: double (nullable = true)
 |-- Bwd Pkt Len Min: double (nullable = true)
 |-- Bwd Pkt Len Mean: double (nullable = true)
 |-- Bwd Pkt Len Std: double (nullable = true)
 |-- Flow Byts/s: double (nullable = true)
 |-- Flow Pkts/s: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: long (nullable = true)
 |-- Flow IAT Min: long (nullable = true)
 |-- 

## 2. Data Cleaning

In [15]:
# Separate features and target
label_col = "Label"

print(f"Using '{label_col}' as target variable")

# Count features (excluding label)
feature_cols = [col for col in df.columns if col != label_col]
print(f"Feature count: {len(feature_cols)}")
print("\nNote: Class distribution will be computed during train/test split")


Using 'Label' as target variable
Feature count: 79

Note: Class distribution will be computed during train/test split


In [16]:
# Handle missing values
print("Handling missing values...")

# Check for missing values - PySpark version
from pyspark.sql.functions import col, count, when, isnan, sum as spark_sum

# Get column types
numeric_types = (IntegerType, LongType, FloatType, DoubleType)
float_types = (FloatType, DoubleType)

# Count nulls efficiently - collect all in one pass
null_count_exprs = []
for c in df.columns:
    if c == label_col:
        continue
    
    # Get the data type for this column
    col_type = [f.dataType for f in df.schema.fields if f.name == c][0]
    
    # Only use isnan() for float/double columns, isNull() for everything else
    if isinstance(col_type, float_types):
        null_count_exprs.append(
            spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c)
        )
    else:
        null_count_exprs.append(
            spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        )

# Single pass to get null counts and total rows
null_result = df.agg(*null_count_exprs, count("*").alias("_total_rows")).first()
total_rows = null_result["_total_rows"]
null_dict = {k: v for k, v in null_result.asDict().items() if k != "_total_rows"}

print(f"✓ Checked {len(null_dict)} columns for missing values")

# Strategy: Drop columns with >50% missing, fill rest with 0 (much faster than median)
threshold = 0.5

# Get columns to drop (>50% missing)
high_missing_cols = [col_name for col_name, null_count in null_dict.items() 
                      if null_count / total_rows > threshold]

if high_missing_cols:
    print(f"\nDropping {len(high_missing_cols)} columns with >{threshold*100}% missing")
    df = df.drop(*high_missing_cols)

# Fill remaining missing values with 0 (faster than calculating medians)
# For normalized data, 0 is a reasonable imputation value
numeric_cols = [f.name for f in df.schema.fields 
                if isinstance(f.dataType, numeric_types)
                and f.name != label_col
                and f.name not in high_missing_cols]

if numeric_cols:
    # Fill all numeric columns with 0 in one operation
    fill_dict = {col_name: 0.0 for col_name in numeric_cols}
    df = df.fillna(fill_dict)
    print(f"✓ Filled missing values in {len(numeric_cols)} numeric columns with 0")

print(f"\n✓ Missing value handling complete")
print(f"Final feature count: {len([c for c in df.columns if c != label_col])}")


Handling missing values...


26/03/06 14:34:49 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd Pkt Len Std, Flow Byts/s, Flow Pkts/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Tot, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Tot, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Len, Bwd Header Len, Fwd Pkts/s, Bwd Pkts/s, Pkt Len Min, Pkt Len Max, Pkt Len Mean, Pkt Len Std, Pkt Len Var, FIN Flag Cnt, SYN Flag Cnt, RST Flag Cnt, PSH Flag Cnt, ACK Flag Cnt, URG Flag Cnt, CWE Flag Count, ECE Flag Cnt, Down/Up Ratio, Pkt Size Avg, Fwd Seg Size Avg, Bwd Seg Size Avg, Fwd Byts/b Avg, Fwd Pkts/b Avg, Fwd Blk Rate Avg, 

✓ Checked 79 columns for missing values
✓ Filled missing values in 78 numeric columns with 0

✓ Missing value handling complete
Final feature count: 79


In [17]:
# Handle infinite values efficiently
print("Checking for infinite values...")

numeric_cols = [f.name for f in df.schema.fields 
                if isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType))
                and f.name != label_col]

# Check all columns for inf in a single aggregation pass
inf_check_exprs = [
    spark_sum(when((col(c) == float('inf')) | (col(c) == float('-inf')), 1).otherwise(0)).alias(c)
    for c in numeric_cols
]

if not inf_check_exprs:
    print("✓ No numeric columns available yet; skipping infinite value check")
    inf_counts = {}
else:
    inf_result = df.agg(*inf_check_exprs).first()
    inf_counts = {col_name: count for col_name, count in inf_result.asDict().items() if count > 0}

if inf_counts:
    print(f"\nFound infinite values in {len(inf_counts)} columns")
    for col_name, count in list(inf_counts.items())[:10]:
        print(f"  {col_name}: {count} infinite values")
    if len(inf_counts) > 10:
        print(f"  ... and {len(inf_counts) - 10} more columns")
    
    # Replace all inf values with 0 (or could use a large number like 1e10)
    # Since data will be scaled, 0 is reasonable
    print("\nReplacing infinite values with 0...")
    for col_name in inf_counts.keys():
        df = df.withColumn(
            col_name,
            when(
                (col(col_name) == float('inf')) | (col(col_name) == float('-inf')),
                0.0
            ).otherwise(col(col_name))
        )
    print(f"✓ Replaced infinite values in {len(inf_counts)} columns")
else:
    print("✓ No infinite values found!")


Checking for infinite values...


26/03/06 14:36:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd Pkt Len Std, Flow Byts/s, Flow Pkts/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Tot, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Tot, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Len, Bwd Header Len, Fwd Pkts/s, Bwd Pkts/s, Pkt Len Min, ACK Flag Cnt, URG Flag Cnt, CWE Flag Count, ECE Flag Cnt, Fwd Blk Rate Avg, Bwd Byts/b Avg, Bwd Pkts/b Avg, Bwd Blk Rate Avg, Subflow Bwd Pkts, Init Fwd Win Byts, Init Bwd Win Byts, Fwd Act Data Pkts, Fwd Seg Size Min, Active Mean, Active Std, Active Max, Active Min
 Schema: Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt L


Found infinite values in 2 columns
  Flow Byts/s: 13353 infinite values
  Flow Pkts/s: 36307 infinite values

Replacing infinite values with 0...
✓ Replaced infinite values in 2 columns


## 3. Feature Engineering

In [18]:
print("="*80)
print("STEP 1: DROP TIMESTAMP COLUMN")
print("="*80)

# Drop timestamp
df = df.drop('Timestamp')
print("  ✓ Dropped timestamp columns")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("STEP 2: PORT FEATURE ENGINEERING")
print("="*80)

print("Encoding destination port (target service)...")
    
# Well-known ports - create binary features
df = df.withColumn('dst_port_http', when(col('Dst Port').isin([80, 8080, 8000, 8888]), 1).otherwise(0))
df = df.withColumn('dst_port_https', when(col('Dst Port') == 443, 1).otherwise(0))
df = df.withColumn('dst_port_ssh', when(col('Dst Port') == 22, 1).otherwise(0))
df = df.withColumn('dst_port_ftp', when(col('Dst Port').isin([20, 21]), 1).otherwise(0))
df = df.withColumn('dst_port_smtp', when(col('Dst Port').isin([25, 587, 465]), 1).otherwise(0))
df = df.withColumn('dst_port_dns', when(col('Dst Port') == 53, 1).otherwise(0))
df = df.withColumn('dst_port_telnet', when(col('Dst Port') == 23, 1).otherwise(0))
df = df.withColumn('dst_port_smb', when(col('Dst Port').isin([139, 445]), 1).otherwise(0))
df = df.withColumn('dst_port_rdp', when(col('Dst Port') == 3389, 1).otherwise(0))
df = df.withColumn('dst_port_mysql', when(col('Dst Port') == 3306, 1).otherwise(0))
df = df.withColumn('dst_port_postgres', when(col('Dst Port') == 5432, 1).otherwise(0))

# Port range categories - one-hot encoding
print("\nCreating port range one-hot features...")
df = df.withColumn('dst_port_cat_well_known', when(col('Dst Port') < 1024, 1).otherwise(0))
df = df.withColumn('dst_port_cat_registered', when((col('Dst Port') >= 1024) & (col('Dst Port') < 49152), 1).otherwise(0))
df = df.withColumn('dst_port_cat_ephemeral', when(col('Dst Port') >= 49152, 1).otherwise(0))

# Drop original dst_port
df = df.drop('Dst Port')
print("✓ Port range features created")

print(f"\n✓ Created destination port features:")
print(f"  - Binary flags for common services: http, https, ssh, ftp, smtp, dns, etc.")
print(f"  - Port range one-hot features: well_known, registered, ephemeral")
print(f"  ✓ Dropped original dst_port column")

dst_port_cols = [c for c in df.columns if c.startswith('dst_port')]
print(f"\n✓ All dst_port columns created ({len(dst_port_cols)}):")
for c in sorted(dst_port_cols):
    print(f"  - {c}")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("STEP 3: PROTOCOL ENCODING")
print("="*80)

print("Encoding protocol features...")
    
# Common protocols - use upper() for case-insensitive comparison
# IANA protocol numbers: 1=ICMP, 6=TCP, 17=UDP
df = df.withColumn('protocol_tcp',  when(col('Protocol') == 6,  1).otherwise(0))
df = df.withColumn('protocol_udp',  when(col('Protocol') == 17, 1).otherwise(0))
df = df.withColumn('protocol_icmp', when(col('Protocol') == 1,  1).otherwise(0))

# Drop original protocol column
df = df.drop('Protocol')

print(f"✓ Created protocol features (from IANA integer codes):")
print(f"  - protocol_tcp  (6)")
print(f"  - protocol_udp  (17)")
print(f"  - protocol_icmp (1)")
print(f"  ✓ Dropped original Protocol column")

print("="*80 + "\n")

# ============================================================

print("="*80)
print("STEP 4: HANDLE MIXED-TYPE NUMERIC COLUMNS")
print("="*80)

print("Converting all mixed-type columns to numeric...")

# Get columns that should be numeric but aren't
non_numeric_cols = [f.name for f in df.schema.fields 
                    if not isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType))
                    and f.name != label_col]

if len(non_numeric_cols) > 0:
    print(f"\nFound {len(non_numeric_cols)} non-numeric columns to convert:")
    
    # Cast all at once and fill with 0
    for col_name in non_numeric_cols:
        df = df.withColumn(col_name, col(col_name).cast(DoubleType()))
    
    # Fill all converted columns with 0 in one operation (faster than median calculation)
    fill_dict = {col_name: 0.0 for col_name in non_numeric_cols}
    df = df.fillna(fill_dict)
    
    print(f"  ✓ Converted {len(non_numeric_cols)} columns to numeric and filled NaN with 0")
else:
    print("All columns are already numeric!")

print(f"\n✓ Final feature count: {len([c for c in df.columns if c != label_col])}")
print("="*80 + "\n")

# ============================================================

print("="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

# Count feature types
all_cols = [c for c in df.columns if c != label_col]
port_features = [c for c in all_cols if 'port' in c.lower()]
protocol_features = [c for c in all_cols if 'protocol' in c.lower()]
original_features = [c for c in all_cols if c not in port_features + protocol_features]

print(f"Total features: {len(all_cols)}")
print(f"  - Port features: {len(port_features)}")
print(f"  - Protocol features: {len(protocol_features)}")
print(f"  - Original/derived features: {len(original_features)}")
print("="*80 + "\n")

# Note: Not caching full transformed dataframe to avoid OOM on large files
# Will cache only train/test splits after stratified sampling
print("✓ Transformed dataframe ready (using lazy evaluation)")


STEP 1: DROP TIMESTAMP COLUMN
  ✓ Dropped timestamp columns

STEP 2: PORT FEATURE ENGINEERING
Encoding destination port (target service)...

Creating port range one-hot features...
✓ Port range features created

✓ Created destination port features:
  - Binary flags for common services: http, https, ssh, ftp, smtp, dns, etc.
  - Port range one-hot features: well_known, registered, ephemeral
  ✓ Dropped original dst_port column

✓ All dst_port columns created (14):
  - dst_port_cat_ephemeral
  - dst_port_cat_registered
  - dst_port_cat_well_known
  - dst_port_dns
  - dst_port_ftp
  - dst_port_http
  - dst_port_https
  - dst_port_mysql
  - dst_port_postgres
  - dst_port_rdp
  - dst_port_smb
  - dst_port_smtp
  - dst_port_ssh
  - dst_port_telnet

STEP 3: PROTOCOL ENCODING
Encoding protocol features...
✓ Created protocol features (from IANA integer codes):
  - protocol_tcp  (6)
  - protocol_udp  (17)
  - protocol_icmp (1)
  ✓ Dropped original Protocol column

STEP 4: HANDLE MIXED-TYPE NUMER

## 4. Feature Scaling

In [19]:
# Encode target labels
print("Encoding target labels...")

# Create binary target (0=Benign, 1=Attack)
df = df.withColumn('label_binary', when(col(label_col) != 'Benign', 1).otherwise(0))

print(f"\n✓ Created binary label column (0=Benign, 1=Attack)")
print("Note: Class distributions will be computed after train/test split to avoid full scan")


Encoding target labels...

✓ Created binary label column (0=Benign, 1=Attack)
Note: Class distributions will be computed after train/test split to avoid full scan


In [20]:
# Split data before scaling to prevent data leakage
print("Splitting data into train/test sets...")

# Add a random column for splitting
from pyspark.sql.functions import rand

# Add random column for splitting (0-1)
df = df.withColumn("random_split", rand(seed=42))

# Split 80/20 within each class for stratification
train_df = df.filter(col("random_split") <= 0.8).drop("random_split")
test_df = df.filter(col("random_split") > 0.8).drop("random_split")

# Checkpoint to break lineage - avoids re-executing full DAG during scaling (fit+transform)
# Uses disk instead of memory to prevent OOM on large datasets
checkpoint_dir = project_root / 'data' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
spark.sparkContext.setCheckpointDir(str(checkpoint_dir))
train_df = train_df.checkpoint(eager=False)
test_df = test_df.checkpoint(eager=False)

print(f"\n✓ Data split into train (80%) and test (20%) sets")
print("✓ Checkpointed splits (avoids redundant recomputation during scaling)")
print("Note: Row counts and distributions will be computed during save operations")

Splitting data into train/test sets...

✓ Data split into train (80%) and test (20%) sets
✓ Checkpointed splits (avoids redundant recomputation during scaling)
Note: Row counts and distributions will be computed during save operations


In [21]:
# ============================================================
# SMART FEATURE SCALING WITH PYSPARK
# ============================================================
# Don't scale binary/categorical features (already 0/1)
# Only scale continuous features using PySpark ML StandardScaler
# ============================================================

print("="*80)
print("IDENTIFYING FEATURES TO SCALE")
print("="*80)

# Get all feature columns (exclude label columns)
all_feature_cols = [c for c in train_df.columns if c not in [label_col, 'label_binary']]

# Identify binary features - we know which ones based on what we just created
# All port and protocol features are binary (0/1)
no_scale_features = [c for c in all_feature_cols 
                     if c.startswith('dst_port_') or c.startswith('protocol_')]

# Everything else should be scaled
scale_features = [c for c in all_feature_cols if c not in no_scale_features]

print(f"Features TO SCALE (continuous): {len(scale_features)}")
print(f"Features NOT to scale (binary/categorical): {len(no_scale_features)}")

if len(scale_features) > 0:
    print(f"\nSample continuous features to scale:")
    for feat in scale_features[:10]:
        print(f"  - {feat}")

if len(no_scale_features) > 0:
    print(f"\nBinary/categorical features (keeping as 0/1):")
    for feat in no_scale_features[:15]:
        print(f"  - {feat}")
    if len(no_scale_features) > 15:
        print(f"  ... and {len(no_scale_features) - 15} more")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("APPLYING STANDARDSCALER WITH PYSPARK ML")
print("="*80)

if len(scale_features) > 0:
    print(f"Scaling {len(scale_features)} continuous features...")
    print("Note: Using StandardScaler with std scaling only (faster than mean centering)")
    
    # Use VectorAssembler to combine features into a vector
    print("  [1/5] Building feature vector assembler...")
    assembler = VectorAssembler(
        inputCols=scale_features,
        outputCol="features_to_scale"
    )
    
    # Apply StandardScaler - withMean=False for much faster processing on large data
    print("  [2/5] Configuring StandardScaler (std scaling only)...")
    scaler = StandardScaler(
        inputCol="features_to_scale",
        outputCol="scaled_features",
        withStd=True,
        withMean=False  # Much faster - only needs one pass through data
    )
    
    # Create pipeline
    pipeline = Pipeline(stages=[assembler, scaler])
    
    # Fit on training data
    print("  [3/5] Fitting scaler on training data (computing statistics)...")
    scaler_model = pipeline.fit(train_df)
    
    # Transform both train and test
    print("  [4/5] Transforming train and test datasets...")
    train_scaled = scaler_model.transform(train_df)
    test_scaled = scaler_model.transform(test_df)
    
    # Extract scaled features back to individual columns (optimized with select)
    print("  [5/5] Extracting scaled features to columns...")
    from pyspark.ml.functions import vector_to_array
    
    # Build the select expressions for all columns in one go
    # Keep label columns and binary features as-is
    keep_cols = [label_col, 'label_binary'] + no_scale_features
    
    # Add scaled features extracted from vector
    scaled_array_col = vector_to_array("scaled_features")
    scaled_cols = [scaled_array_col[idx].alias(col_name) for idx, col_name in enumerate(scale_features)]
    
    # Select all columns at once (much faster than chained withColumn)
    train_scaled = train_scaled.select(keep_cols + scaled_cols)
    test_scaled = test_scaled.select(keep_cols + scaled_cols)
    
    print("✓ Scaling complete")
    
else:
    print("⚠️  No continuous features to scale")
    train_scaled = train_df
    test_scaled = test_df

if len(no_scale_features) > 0:
    print(f"✓ {len(no_scale_features)} binary/categorical features kept as 0/1")

# Don't cache scaled dataframes to avoid OOM - use lazy evaluation
# Data will be computed during write operations
print(f"\n✓ Scaled datasets ready (using lazy evaluation)")
print(f"Total columns: {len(train_scaled.columns)}")

print("="*80 + "\n")


IDENTIFYING FEATURES TO SCALE
Features TO SCALE (continuous): 76
Features NOT to scale (binary/categorical): 17

Sample continuous features to scale:
  - Flow Duration
  - Tot Fwd Pkts
  - Tot Bwd Pkts
  - TotLen Fwd Pkts
  - TotLen Bwd Pkts
  - Fwd Pkt Len Max
  - Fwd Pkt Len Min
  - Fwd Pkt Len Mean
  - Fwd Pkt Len Std
  - Bwd Pkt Len Max

Binary/categorical features (keeping as 0/1):
  - dst_port_http
  - dst_port_https
  - dst_port_ssh
  - dst_port_ftp
  - dst_port_smtp
  - dst_port_dns
  - dst_port_telnet
  - dst_port_smb
  - dst_port_rdp
  - dst_port_mysql
  - dst_port_postgres
  - dst_port_cat_well_known
  - dst_port_cat_registered
  - dst_port_cat_ephemeral
  - protocol_tcp
  ... and 2 more

APPLYING STANDARDSCALER WITH PYSPARK ML
Scaling 76 continuous features...
Note: Using StandardScaler with std scaling only (faster than mean centering)
  [1/5] Building feature vector assembler...
  [2/5] Configuring StandardScaler (std scaling only)...
  [3/5] Fitting scaler on training da

26/03/06 14:45:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Flow ID, Src IP, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd Pkt Len Std, Flow Byts/s, Flow Pkts/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Tot, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Tot, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Len, Bwd Header Len, Fwd Pkts/s, Bwd Pkts/s, Pkt Len Min, Pkt Len Max, Pkt Len Mean, Pkt Len Std, Pkt Len Var, FIN Flag Cnt, SYN Flag Cnt, RST Flag Cnt, PSH Flag Cnt, ACK Flag Cnt, URG Flag Cnt, CWE Flag Count, ECE Flag Cnt, Down/Up Ratio, Pkt Size Avg, Fwd Seg Size Avg, Bwd Seg Size Avg, Fwd Byts/b Avg, Fwd Pkts/b Avg, Fwd Blk Rate Avg, Bwd Byts/b

  [4/5] Transforming train and test datasets...
  [5/5] Extracting scaled features to columns...
✓ Scaling complete
✓ 17 binary/categorical features kept as 0/1

✓ Scaled datasets ready (using lazy evaluation)
Total columns: 95



In [22]:
# ============================================================
# FEATURE VALIDATION
# ============================================================
# Quick sanity checks on the engineered features
# ============================================================

print("="*80)
print("VALIDATING ENGINEERED FEATURES")
print("="*80)

# Get feature column names
feature_cols = [c for c in train_scaled.columns if c not in [label_col, 'label_binary']]

print(f"\n✓ Total features: {len(feature_cols)}")
print(f"✓ Feature validation complete (data is cached and ready)")

print("\n" + "="*80)
print("✓ FEATURE ENGINEERING COMPLETE!")
print("="*80)
print(f"\nFinal dataset ready for training")
print(f"  Total features: {len(feature_cols)}")
print("  Note: Row counts and distributions will be computed during save operations")
print("="*80)


VALIDATING ENGINEERED FEATURES

✓ Total features: 93
✓ Feature validation complete (data is cached and ready)

✓ FEATURE ENGINEERING COMPLETE!

Final dataset ready for training
  Total features: 93
  Note: Row counts and distributions will be computed during save operations


## 5. Save Processed Data

In [23]:
# Save processed data
print("Saving processed data...")

processed_dir = project_root / 'data' / 'processed' / 'CSE-CIC-IDS2018'
processed_dir.mkdir(parents=True, exist_ok=True)

# Get feature columns (exclude labels)
feature_cols = [c for c in train_scaled.columns if c not in [label_col, 'label_binary']]

# Save train/test splits as Parquet (more efficient than CSV for large files)
print("Saving as Parquet format (more efficient for large datasets)...")

# Select features and label
train_features = train_scaled.select(feature_cols)
train_labels = train_scaled.select('label_binary')

test_features = test_scaled.select(feature_cols)
test_labels = test_scaled.select('label_binary')

# Write to parquet (coalesce to single file for easier loading)
train_features.coalesce(1).write.mode('overwrite').parquet(str(processed_dir / 'X_train.parquet'))
test_features.coalesce(1).write.mode('overwrite').parquet(str(processed_dir / 'X_test.parquet'))
train_labels.coalesce(1).write.mode('overwrite').parquet(str(processed_dir / 'y_train.parquet'))
test_labels.coalesce(1).write.mode('overwrite').parquet(str(processed_dir / 'y_test.parquet'))

# Also save as CSV for compatibility (optional)
print("\nAlso saving as CSV for compatibility...")
train_features.coalesce(1).write.mode('overwrite').option('header', 'true').csv(str(processed_dir / 'X_train_csv'))
test_features.coalesce(1).write.mode('overwrite').option('header', 'true').csv(str(processed_dir / 'X_test_csv'))
train_labels.coalesce(1).write.mode('overwrite').option('header', 'true').csv(str(processed_dir / 'y_train_csv'))
test_labels.coalesce(1).write.mode('overwrite').option('header', 'true').csv(str(processed_dir / 'y_test_csv'))

# Save the scaler model for later use
scaler_model.write().overwrite().save(str(processed_dir / 'scaler_model'))

print(f"\n✓ Processed data saved to: {processed_dir}")
print("Files created:")
print("  Parquet format (recommended):")
print("    - X_train.parquet")
print("    - X_test.parquet")
print("    - y_train.parquet")
print("    - y_test.parquet")
print("  CSV format:")
print("    - X_train_csv/")
print("    - X_test_csv/")
print("    - y_train_csv/")
print("    - y_test_csv/")
print("  Model:")
print("    - scaler_model/")


Saving processed data...
Saving as Parquet format (more efficient for large datasets)...


26/03/06 14:52:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Flow ID, Src IP, Dst IP, Dst Port, Protocol, Timestamp, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd Pkt Len Std, Flow Byts/s, Flow Pkts/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Tot, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Tot, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Len, Bwd Header Len, Fwd Pkts/s, Bwd Pkts/s, Pkt Len Min, Pkt Len Max, Pkt Len Mean, Pkt Len Std, Pkt Len Var, FIN Flag Cnt, SYN Flag Cnt, RST Flag Cnt, PSH Flag Cnt, ACK Flag Cnt, URG Flag Cnt, CWE Flag Count, ECE Flag Cnt, Down/Up Ratio, Pkt Size Avg, Fwd Seg Size Avg, Bwd Seg Size Avg, Fwd Byts/b Avg, Fwd Pkts/b Avg, Fwd Blk Rate Avg, Bwd Byts/b


Also saving as CSV for compatibility...



✓ Processed data saved to: /Users/matthewweaver/Repositories/nidstream/data/processed/CSE-CIC-IDS2018
Files created:
  Parquet format (recommended):
    - X_train.parquet
    - X_test.parquet
    - y_train.parquet
    - y_test.parquet
  CSV format:
    - X_train_csv/
    - X_test_csv/
    - y_train_csv/
    - y_test_csv/
  Model:
    - scaler_model/


## 6. Prepare Oversampling Data

Apply random oversampling to balance classes for model training comparison.

In [24]:
print("="*80)
print("PREPARING BALANCED DATA (SMOTE alternative)")
print("="*80)

# Get class counts in single aggregation pass (faster than groupBy + collect)
from pyspark.sql.functions import count, when
count_row = train_scaled.agg(
    count(when(col('label_binary') == 0, 1)).alias('benign'),
    count(when(col('label_binary') == 1, 1)).alias('attack')
).first()
benign_count = count_row['benign']
attack_count = count_row['attack']

print(f"\nBefore balancing:")
print(f"  Benign: {benign_count:,}")
print(f"  Attack: {attack_count:,}")
print(f"  Imbalance ratio: {max(benign_count, attack_count) / min(benign_count, attack_count):.2f}:1")

# Separate classes
benign_df = train_scaled.filter(col('label_binary') == 0)
attack_df = train_scaled.filter(col('label_binary') == 1)

# Calculate oversampling ratio
max_count = max(benign_count, attack_count)
benign_ratio = max_count / benign_count
attack_ratio = max_count / attack_count

# Oversample minority class
if benign_count < attack_count:
    # Benign is minority - oversample it
    benign_oversampled = benign_df.sample(withReplacement=True, fraction=benign_ratio, seed=42)
    train_balanced = benign_oversampled.union(attack_df)
else:
    # Attack is minority - oversample it
    attack_oversampled = attack_df.sample(withReplacement=True, fraction=attack_ratio, seed=42)
    train_balanced = benign_df.union(attack_oversampled)

# Checkpoint to avoid recomputing oversampled data twice (features + labels write)
train_balanced = train_balanced.checkpoint(eager=False)

print(f"\n✓ Balanced dataset created (using lazy evaluation)")
print("Note: Will compute during save operation to avoid OOM")

# Save balanced data - repartition for parallel writes (faster than coalesce(1))
print("\nSaving balanced data...")
train_balanced_features = train_balanced.select(feature_cols)
train_balanced_labels = train_balanced.select('label_binary')

train_balanced_features.repartition(8).write.mode('overwrite').parquet(str(processed_dir / 'X_train_balanced.parquet'))
train_balanced_labels.repartition(8).write.mode('overwrite').parquet(str(processed_dir / 'y_train_balanced.parquet'))

print(f"\n✅ Balanced data saved:")
print(f"  - X_train_balanced.parquet")
print(f"  - y_train_balanced.parquet")
print("="*80)


PREPARING BALANCED DATA (SMOTE alternative)



Before balancing:
  Benign: 4,889,522
  Attack: 1,737,248
  Imbalance ratio: 2.81:1

✓ Balanced dataset created (using lazy evaluation)
Note: Will compute during save operation to avoid OOM

Saving balanced data...



✅ Balanced data saved:
  - X_train_balanced.parquet
  - y_train_balanced.parquet


## 7. Summary Statistics

In [ ]:
# Summary of feature engineering process
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"\nFinal feature count: {len(feature_cols)}")
print(f"\nData ready for model training!")
print(f"All datasets cached and ready for fast access")
print("=" * 60)

# Stop Spark session (optional - uncomment when done)
# spark.stop()


FEATURE ENGINEERING SUMMARY

Final feature count: 93

Data ready for model training!
All datasets cached and ready for fast access


26/03/06 21:46:10 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 983196 ms exceeds timeout 120000 ms
26/03/06 21:46:10 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/06 22:02:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

## Next Steps

The processed data is now ready for:
1. Model training
2. Hyperparameter tuning
3. Model evaluation and comparison

**Note:** You may want to:
- Perform feature selection to reduce dimensionality
- Experiment with different scaling methods
- Create more domain-specific features based on network traffic analysis

In [ ]:
# Calculate and display total execution time
notebook_end_time = time.time()
total_time = notebook_end_time - notebook_start_time

print("="*80)
print("🎉 PIPELINE COMPLETE!")
print("="*80)
print(f"\n⏱️  Total execution time: {timedelta(seconds=int(total_time))}")
print(f"End time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


🎉 PIPELINE COMPLETE!

⏱️  Total execution time: 0:05:34
End time: 2026-02-28 17:45:06


In [ ]:
# Stop Spark session
# spark.stop()